## Cosmic Web - CuPy - Array Model

[CuPy](https://docs.cupy.dev/en/stable/) runs the reference, NumPy expressions over device arrays, on the GPU: the *array model*. Every expression launches at least one kernel and, unless it writes in place, allocates a new array. In this notebook we trace one call of the reference, count what each line costs, then remove the temporaries with views and buffers.

### Table of Contents

1. [Setup](#1-setup)
2. [Stencil](#2-stencil)
3. [Histogram](#3-histogram)
4. [Trace](#4-trace)
5. [Bytes model](#5-bytes-model)
6. [Views and buffers](#6-views-and-buffers)
7. [Limitations and recap](#7-limitations-and-recap)

---

### 1. Setup

The binned field loads from the intro's cache under `data/`. The intro's curve is the reference.

---

**Quick Docs**

- `cw.setup(stage, note)`: `(N, K)` from `machine.json`, after checking the device has room for the stage at that size; prints the field's size and the device, then `note`.

In [ ]:
import cupy as cp

import cosmic_core as cw

N, K = cw.setup('cupy', f'CuPy {cp.__version__}')

### 2. Stencil

On whole arrays, each birth is computed with one `cp.roll` and one `cp.minimum`. Every line allocates: both functions return a new array.


In [ ]:
def ecc_cells_cupy(q):
    '''Births of the eight cells each voxel owns: (vertex, (ex, ey, ez), (fxy, fxz, fyz), cube).'''
    ex = cp.minimum(q, cp.roll(q, -1, 0))
    ey = cp.minimum(q, cp.roll(q, -1, 1))
    ez = cp.minimum(q, cp.roll(q, -1, 2))
    fxy = cp.minimum(ex, cp.roll(ex, -1, 1))
    fxz = cp.minimum(ex, cp.roll(ex, -1, 2))
    fyz = cp.minimum(ey, cp.roll(ey, -1, 2))
    c = cp.minimum(fxy, cp.roll(fxy, -1, 2))
    return q, (ex, ey, ez), (fxy, fxz, fyz), c

### 3. Histogram

The eight births of every voxel are counted into one signed histogram. [`cp.bincount`](https://docs.cupy.dev/en/stable/reference/generated/cupy.bincount.html) counts one array at a time, so the histogram is the signed sum of eight counts. Its docs warn that it may synchronise the device.

---

**Quick Docs**

- `cp.bincount(x, minlength=K)`: counts of each value in `x` as int64; the output length is the larger of `x.max() + 1` and `K`.


In [ ]:
def signed_hist(cells, k=K):
    '''Signed histogram of the eight cell arrays, int64 of length k: +1 per vertex and face, -1 per edge and cube.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = cells
    bc = lambda a: cp.bincount(a.ravel(), minlength=k)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)


def ecc_hist_cupy(q, k=K):
    return signed_hist(ecc_cells_cupy(q), k)


def run_cupy(n):
    hist = ecc_hist_cupy(cw.field(n))
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist)

Now let's make sure it works. The first call is the cold one: CuPy compiles each kernel on its first use and makes the first allocations.

In [ ]:
warm = cw.measure('cupy', run_cupy, N, record=False)  # the intro already recorded the cupy stage

### 4. Trace

How many kernels did that call launch, and where do its copies come from? [Nsight Systems](https://docs.nvidia.com/nsight-systems/) records every GPU operation of a process: the *trace*. `cw.trace_call` writes a self-contained *probe* script of the implementation above (the profiler runs a fresh process) and traces one call of it.

CuPy caches freed device memory in a *memory pool* and reuses it for later allocations. The pool's growth over one call is the most memory the call held at once.

---

**Quick Docs**

- `cw.trace_call(stage, run, n, functions, state, header)`: writes `nsys_probe_<stage>.py` from the source of `run`, `functions` and the `state` lines, traces one call, prints its kernels, copies and memsets and attaches them to the stage file.
- `cp.get_default_memory_pool()`: `.free_all_blocks()` releases what is unused, `.total_bytes()` is what the pool holds.


In [ ]:
trace = cw.trace_call('cupy', run_cupy, N, [ecc_cells_cupy, signed_hist, ecc_hist_cupy])

pool = cp.get_default_memory_pool()
pool.free_all_blocks()  # release unused blocks, so the pool's growth below is one call's peak memory
base = pool.total_bytes()
run_cupy(N)
pool_arrays = (pool.total_bytes() - base) / cw.field(N).nbytes
print(f'one call added {pool_arrays:.1f} full-size arrays to the memory pool')

- One call launches tens of kernels: every `cp.roll`, `cp.minimum` and `cp.bincount` runs at least one.
- `cp.bincount` checks the sign of its input and takes its maximum to size the output. Both are scalars the Python code needs, so each is a device-to-host copy that synchronises: most of the memcpys.
- `cp.bincount` zeroes its output before counting: the memsets.


### 5. Bytes model

Let's build the *bytes model*: the bytes each operation moves through device memory, summed over the implementation as written:

- Every roll reads the field and writes a copy.
- Every minimum reads two arrays and writes one.
- Every bincount reads its array three times: the sign check (also writes and reads a one-byte temporary), the maximum that sizes the output, and the count.

---

**Quick Docs**

- `cw.roof_report(tool, record)`: the operations, counts and bytes of the implementation as written, then converts the total into the time a device copy would need for those bytes and sets it beside the record's warm time.


In [ ]:
cw.roof_report('cupy', warm)

- The model and the measurement are close: the arrays are far larger than the caches, so every byte the model counts moves.


### 6. Views and buffers

Two changes remove the temporaries within the array model:

- A periodic shift can be written as two slice pairs: `a[:-1]` against `a[1:]`, then the last plane against the first. Slices are views: they share the array's memory.
- `cp.minimum(a, b, out=buf)` writes into a preallocated buffer.

The TODO writes `min_shift`, the shifted minimum along one axis through views. The stencil built from it, the gate and the timing are written.

---

**Quick Docs**

- `a[lo]`, `a[hi]`: views; `lo` and `hi` are index tuples with the slices along `axis` and `slice(None)` elsewhere.
- `cp.minimum(a, b, out=buf)`: the elementwise minimum written into `buf`, which may be a view.
- `cw.by_size(build)`: memoise `build(n)`; passed as `prepare`, it allocates the buffers before the timer starts.

In [ ]:
def min_shift(a, axis, out):
    '''Write minimum(a, a shifted by -1 along axis), periodic, into out through views.'''
    at = lambda s: tuple(s if d == axis else slice(None) for d in range(3))  # the index tuple with s along axis, full slices elsewhere
    lo, hi, last, first = at(slice(None, -1)), at(slice(1, None)), at(-1), at(0)
    # TODO: two cp.minimum(..., out=) calls: a[lo] against a[hi] into out[lo], then the wrap plane a[last] against a[first] into out[last].
    ...

In [ ]:
def ecc_cells_cupy_views(q, bufs):
    '''The eight cell arrays, every minimum written into a preallocated buffer.'''
    ex, ey, ez, fxy, fxz, fyz, c = bufs
    min_shift(q, 0, ex)
    min_shift(q, 1, ey)
    min_shift(q, 2, ez)
    min_shift(ex, 1, fxy)
    min_shift(ex, 2, fxz)
    min_shift(ey, 2, fyz)
    min_shift(fxy, 2, c)
    return q, (ex, ey, ez), (fxy, fxz, fyz), c


def ecc_hist_cupy_views(q, bufs, k=K):
    return signed_hist(ecc_cells_cupy_views(q, bufs), k)


BUFFERS = cw.by_size(lambda n: [cp.empty_like(cw.field(n)) for _ in range(7)])


def run_cupy_views(n):
    hist = ecc_hist_cupy_views(cw.field(n), BUFFERS(n))
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist)

Now let's make sure it works.

In [ ]:
_ = cw.measure('cupy_views', run_cupy_views, N, prepare=BUFFERS)

What did the views remove?


In [ ]:
views = cw.trace_call('cupy_views', run_cupy_views, N, [min_shift, ecc_cells_cupy_views, signed_hist, ecc_hist_cupy_views], state=[BUFFERS])
print(f"with roll: {trace['kernels']} kernels, {trace['memcpy']} memcpys")

pool.free_all_blocks()
base = pool.total_bytes()
run_cupy_views(N)
print(f'one call added {(pool.total_bytes() - base) / cw.field(N).nbytes:.1f} full-size arrays to the memory pool ({pool_arrays:.1f} with roll)')
cw.plot_timelines({'roll': trace['rows'], 'views': views['rows']}, title=f'one call at N = {N}^3: with roll against views')
cw.save_sweep('cupy_views', run_cupy_views)

- The roll temporaries are gone. Each `min_shift` launches two minimum kernels, so the kernel count falls by less than the number of copies removed.
- On the timeline the stencil (the solid block of kernels before the first copy) is far shorter. The histogram stretch, with its copies and memsets, kept its length.
- The eight `cp.bincount` passes and their host waits remain: the histogram is where the array model stops.


Takeaways:

- `cp.roll` copies its input. The same periodic shift written as two slice pairs reads through views and does not perform any copies.
- `cp.bincount` checks the sign of its input and sizes its output from its maximum, and reads both results back to the host: two synchronisations per call, even with `minlength`.

### 7. Limitations and recap

The array model launches at least one kernel per expression, in program order, over whole arrays. Each operation runs as its own kernel: every minimum is written to memory in full before the histogram reads it.

Recap:

- The reference is NumPy expressions over device arrays. The trace counts its launches and shows where the host waits.

Next: `92__cosmic_web__jax__fusion.ipynb` traces the same code once and lets JAX fuse the chain into a few kernels.
